# Week 3 on a GPU: Does ReAct Beat Act-Only With Bigger Models?

<a href="https://colab.research.google.com/github/jorshimayor/AI-Engineering-Flow_Research/blob/main/jupyter/week3_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Week 3 ran the hand-written ReAct agent with **Qwen2.5-0.5B** on a laptop CPU. ReAct scored 20% and Act-only 23%, too close to call, and the 1.5B model was too slow to run locally (about 1.3 tokens/s).

This notebook reruns the **same 30 problems and four modes** (direct, CoT, Act-only, ReAct) with **Qwen2.5-1.5B-Instruct** and **Qwen2.5-3B-Instruct** on a free GPU, then compares all three sizes.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (free tier), then `Runtime → Run all`.

| Step | Time on a T4 (rough estimate) |
|---|---|
| Setup | 1 to 2 min |
| 1.5B model, 120 runs | about 15 to 25 min |
| 3B model, 120 runs | about 25 to 45 min |

Results are written one problem at a time. If Colab disconnects, re-run the cells: finished problems are skipped. With **Google Drive** turned on (below), the results also survive a full runtime reset.

*Also works on Kaggle:* new notebook → `Settings → Accelerator → GPU T4` and `Internet → On`, then import this file.

## 1. Check the GPU

In [ ]:
import torch, subprocess
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB",
          "| bf16 supported:", torch.cuda.is_bf16_supported())
else:
    print("No GPU: Runtime → Change runtime type → T4 GPU, then run again.")

## 2. Get the code

Clones the repo and installs the same `transformers` version used for the local Week 3 run, so results are comparable. Colab's own PyTorch with CUDA is kept.

In [ ]:
REPO = "https://github.com/jorshimayor/AI-Engineering-Flow_Research.git"
BRANCH = "main"
import os
# Always use the latest code: clone if missing, otherwise update in place (a reused runtime keeps the old folder)
if not os.path.exists("AI-Engineering-Flow_Research"):
    !git clone --depth 1 --branch {BRANCH} {REPO}
%cd /content/AI-Engineering-Flow_Research
!git fetch -q --depth 1 origin {BRANCH} && git checkout -q -B {BRANCH} FETCH_HEAD
!pip install -q "transformers==4.49.0" accelerate
!git log --oneline -1

## 3. Keep results on Google Drive (recommended)

Set `USE_DRIVE = True` to store results in `MyDrive/AI-Engineering/week3_gpu`. You'll be asked to authorise Drive access once. With `False`, results live in the runtime and are lost if it resets, so download them in step 6.

In [ ]:
USE_DRIVE = True
RESULTS = "results/week3"
if USE_DRIVE:
    import shutil
    from google.colab import drive
    drive.mount("/content/drive")
    target = "/content/drive/MyDrive/AI-Engineering/week3_gpu"
    os.makedirs(target, exist_ok=True)
    if os.path.isdir(RESULTS) and not os.path.islink(RESULTS):   # keep files the repo already has here
        for f in os.listdir(RESULTS):
            if os.path.isfile(os.path.join(RESULTS, f)) and not os.path.exists(os.path.join(target, f)):
                shutil.copy(os.path.join(RESULTS, f), target)
        shutil.rmtree(RESULTS)
    os.makedirs(os.path.dirname(RESULTS), exist_ok=True)
    if not os.path.islink(RESULTS):
        os.symlink(target, RESULTS)
os.makedirs(RESULTS, exist_ok=True)
print("results go to:", os.path.realpath(RESULTS))

## 4. Run the benchmark

Each command runs all 4 modes on the 30 Week 3 problems and prints a summary with 95% confidence intervals and a paired ReAct vs Act-only test. Re-running a cell resumes where it stopped.

In [ ]:
!python src/run_react_bench.py --model Qwen/Qwen2.5-1.5B-Instruct

In [ ]:
!python src/run_react_bench.py --model Qwen/Qwen2.5-3B-Instruct

## 5. Compare model sizes

The 0.5B results come from the original CPU run (`results/week3/Qwen2.5-0.5B-Instruct.jsonl`, already in the repo).

In [ ]:
import json, sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
sys.path.insert(0, "src")
from stats import wilson_ci, discordant, mcnemar_exact, paired_bootstrap_ci

FILES = {"0.5B": Path("results/week3/Qwen2.5-0.5B-Instruct.jsonl"),
         "1.5B": Path(RESULTS) / "Qwen2.5-1.5B-Instruct.jsonl",
         "3B":   Path(RESULTS) / "Qwen2.5-3B-Instruct.jsonl"}
MODES = ["direct", "cot", "act", "react"]
data = {size: [json.loads(l) for l in f.open()] for size, f in FILES.items() if f.exists()}

fig, ax = plt.subplots(figsize=(9, 4))
w = 0.2
for i, mode in enumerate(MODES):
    xs, ys, err = [], [], [[], []]
    for j, (size, R) in enumerate(data.items()):
        rs = [r for r in R if r["mode"] == mode]
        if not rs:
            continue
        k, n = sum(r["correct"] for r in rs), len(rs)
        lo, hi = wilson_ci(k, n)
        xs.append(j + (i - 1.5) * w); ys.append(k / n * 100); err[0].append((k / n - lo) * 100); err[1].append((hi - k / n) * 100)
    ax.bar(xs, ys, w, yerr=err, capsize=3, label=mode)
ax.set_xticks(range(len(data))); ax.set_xticklabels([f"Qwen2.5-{s}" for s in data])
ax.set_ylabel("accuracy on 30 problems (%)"); ax.set_ylim(0, 105); ax.legend(ncol=4)
ax.set_title("Week 3 benchmark by model size (95% Wilson CIs)")
plt.tight_layout(); plt.savefig(Path(RESULTS) / "accuracy_by_size.png", dpi=130); plt.show()

print(f"{'size':5} {'ReAct':>6} {'Act':>6} {'ReAct - Act (95% CI)':>26} {'McNemar p':>10} {'tokens ReAct/Act':>17}")
for size, R in data.items():
    a = [r["correct"] for r in sorted(R, key=lambda r: r["pid"]) if r["mode"] == "react"]
    b = [r["correct"] for r in sorted(R, key=lambda r: r["pid"]) if r["mode"] == "act"]
    if len(a) != len(b) or not a:
        continue
    eff, lo, hi = paired_bootstrap_ci(a, b)
    tok = lambda m: np.mean([r["tokens_in"] + r["tokens_out"] for r in R if r["mode"] == m])
    print(f"{size:5} {np.mean(a):6.1%} {np.mean(b):6.1%} {eff*100:+8.1f} ({lo*100:+.1f} to {hi*100:+.1f}) "
          f"{mcnemar_exact(*discordant(a, b)):10.3f} {tok('react') / tok('act'):17.2f}")

**Reading the result honestly (from the Week 4 audit):** 30 problems give low statistical power. A ReAct vs Act gap under about 15 to 20 points can't be distinguished from noise here. Treat this as a pilot. In particular, the share of problems where the two modes disagree (the *discordance*) is the number that sizes the full Week 4 protocol (`notes/week_4/eval_protocol_v1.3.md`).

To run more problems with the same templates: `!python src/run_react_bench.py --model ... --n-per 50 --out results/week3/big.jsonl` (300 problems). That isn't the full protocol, which also needs held-out templates the prompt never showed.

## 6. Save the results

In [ ]:
!cd results && zip -qr /content/week3_results.zip week3/
from google.colab import files
files.download("/content/week3_results.zip")

Then on your laptop, unzip into `results/` in the repo and commit. With Drive enabled, the files are also in `MyDrive/AI-Engineering/week3_gpu`.